# 04 — Inventário de metano CAMS73

Este notebook consulta somente o manifesto CH4 local e confere SHA-256 das partições processadas. Não baixa dados nem escreve no manifesto. Um ano completo exige todos os dias dos seus 12 meses.

In [ ]:
NOTEBOOK = "04_ch4_inventory"
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/config.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Execute dentro do projeto que contém config/config.yaml.")
sys.path.insert(0, str(ROOT / "notebooks"))
from cams_air_quality.config import load_config
from cams_air_quality import ghg_inversion as ghg
from cams_air_quality.extraction import save_table
from available_ch4 import (inventory, open_level, coverage, global_series,
                           point_series, monthly_series, annual_series,
                           reference_cycle, anomaly_series, selected_area)

CONFIG_PATH = Path(os.environ.get("CAMSAQ_CONFIG", ROOT / "config/config.yaml"))
cfg = load_config(CONFIG_PATH)
LEVEL = 1
START = None
END = None
BASELINE_START = None
BASELINE_END = None
BBOX = None
LAT = -23.5505
LON = -46.6333
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path(cfg["root"]) / "reports" / "automatic_notebooks" / NOTEBOOK / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
errors = []
print("Saídas:", OUT)

In [ ]:
rows, table = inventory(cfg)
table.to_csv(OUT / "inventory.csv", index=False)
ready = table.loc[table.status == "ready"].to_dict("records") if not table.empty else []
info = coverage(ready)
info.update(source="GHG_INVERSION", release=cfg["ghg_inversion"]["version"],
            files=[row["path"] for row in ready])
(OUT / "coverage.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(OUT / "analysis.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(OUT / "errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")
if not ready:
    print("Nenhum mês CH4 processado e íntegro. Execute camsaq ch4 download ou import e process.")
else:
    print("Meses completos:", info["complete_months"])
    print("Anos completos:", info["complete_years"])
    print("Meses faltantes para média anual:", info["months_needed_for_annual"])
display(table)
print("Erros:", len(errors))

`inventory.csv` mostra estado, caminho, checksum, dias e níveis de cada mês. `coverage.json` lista lacunas e meses necessários para fechar cada ano. A cobertura local não demonstra que toda a série do ADS foi adquirida.